In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [2]:
from langgraph_starter import *
from typing import Optional

/Users/kara/Desktop/langgraphfcc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:

# Give input topic
# LLM write tweet
# Tweet goes to reviewer node
# Reviewer node approves or rejects
# If approved, goes to poster node to post on Twitter
# If rejected, goes back to LLM with feedback to rewrite

In [4]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    decision: Literal["approved", "rejected"]
    feedback: Optional[str]
    iteration: int

class Review(BaseModel):
    decision: Literal["approved", "rejected"]
    feedback: Optional[str] = None


In [5]:
def generate_tweet(state: TweetState) -> TweetState:

    prompt = f"""write a tweet about {state["topic"]} in 280 characters or less. The tone of tweet should be witty with virality factor. 
    Return only the tweet, without any additional text or formatting."""

    tweet_text = llm.invoke(prompt).content.strip()
    
    return {"tweet": tweet_text}

def review_tweet(state: TweetState) -> TweetState:

    prompt = f"""You are a Twitter content reviewer. Review the following tweet and determine if it is suitable for posting on Twitter. 
    Consider factors such as relevance, engagement potential, and adherence to Twitter's content guidelines. 
    Return a JSON result with:
    - decision: "approved" or "rejected"
    - feedback: null if approved, otherwise specific feedback

    Tweet: {state["tweet"]}"""
    review = llm.with_structured_output(Review).invoke(prompt)
    feedback = None if review.decision == "approved" else review.feedback
    return {"decision": review.decision, "feedback": feedback}

def rewrite_tweet(state: TweetState) -> TweetState:

    prompt = f"""The following tweet was rejected by the reviewer with this feedback: {state["feedback"]}. 
    Rewrite the tweet to address the feedback and make it suitable for posting on Twitter. 
    Return only the revised tweet, without any additional text or formatting.

    Original Tweet: {state["tweet"]} and topic: {state["topic"]}"""

    revised_tweet = llm.invoke(prompt).content.strip()
    
    return {"tweet": revised_tweet}


In [6]:
graph = StateGraph(TweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("review_tweet", review_tweet)
graph.add_node("rewrite_tweet", rewrite_tweet)

graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "review_tweet")
graph.add_edge("rewrite_tweet", "review_tweet")

def route_review(state: TweetState) -> str:
    # Must return one of the keys in the mapping below
    return state.get("decision", "rejected")

graph.add_conditional_edges(
    "review_tweet",
    route_review,
    {
        "approved": END,
        "rejected": "rewrite_tweet",
    },
)

workflow = graph.compile()

In [7]:
initial_state = {'topic': "key"}

finalTweet = workflow.invoke(initial_state)

In [8]:
print(finalTweet['tweet'])

"Warning: key holders, beware of a life of turning, unlocking, and constantly being asked to 'key' in your passwords #KeyLifeProblems #PasswordPains"
